# 高精度でなかった国（65ヶ国に入らなかった国）の分析とグループ分け

予測精度が高くなかった国＝65ヶ国リストに含まれていない国を対象に分析します。

**学会発表用の解釈：** 「精度が出なかった」のは偶然ではなく、**既存のモデルでは扱いづらい特有の事情**を抱えた国々である。これらを5つのグループに分類し、各グループで**共通するデータ特性**（観測数・収量ばらつき・気候変数など）を可視化することで、「特異点（Outliers）であり一般的な回帰モデルには馴染まない」と主張する材料とする。

In [11]:
import pandas as pd
import numpy as np
import os

base_dir = r'C:\Users\xyz19\OneDrive\デスクトップ\予測モデルデータセット'
regressionDF = pd.read_excel(os.path.join(base_dir, 'regression_df_20260112_233447.xlsx'))
opt65 = pd.read_csv(os.path.join(base_dir, 'optimal_crop_65countries_5scenarios_rice088.csv'))

areas_65 = sorted(opt65['Area'].unique().tolist())
areas_all = sorted(regressionDF['Area'].unique().tolist())
areas_low = sorted(set(areas_all) - set(areas_65))  # 高精度でなかった国

print(f"回帰データの国数: {len(areas_all)}")
print(f"高精度だった国（65ヶ国）: {len(areas_65)}")
print(f"高精度でなかった国: {len(areas_low)} ヶ国")
print("\n【高精度でなかった国のリスト】")
for i, a in enumerate(areas_low, 1):
    print(f"  {i:2d}. {a}")

回帰データの国数: 156
高精度だった国（65ヶ国）: 65
高精度でなかった国: 91 ヶ国

【高精度でなかった国のリスト】
   1. Afghanistan
   2. Albania
   3. Algeria
   4. Angola
   5. Antigua and Barbuda
   6. Azerbaijan
   7. Barbados
   8. Belize
   9. Benin
  10. Bhutan
  11. Bosnia and Herzegovina
  12. Botswana
  13. Brazil
  14. Brunei Darussalam
  15. Bulgaria
  16. Burkina Faso
  17. Cabo Verde
  18. Cambodia
  19. Cameroon
  20. Central African Republic
  21. Chad
  22. Congo
  23. Croatia
  24. Cuba
  25. Cyprus
  26. Côte d'Ivoire
  27. Denmark
  28. Djibouti
  29. Dominica
  30. Eritrea
  31. Estonia
  32. Ethiopia
  33. Finland
  34. Gabon
  35. Georgia
  36. Grenada
  37. Guinea
  38. Hungary
  39. India
  40. Indonesia
  41. Iraq
  42. Ireland
  43. Jamaica
  44. Jordan
  45. Kenya
  46. Lao People's Democratic Republic
  47. Latvia
  48. Lebanon
  49. Lesotho
  50. Liberia
  51. Lithuania
  52. Luxembourg
  53. Madagascar
  54. Malawi
  55. Malaysia
  56. Maldives
  57. Mali
  58. Malta
  59. Mauritius
  60. Mongolia
  61

## 地域マッピング（グループ分け用）

全国を北米・中南米・オセアニア・アジア・ヨーロッパ・アフリカ・中東に分類し、未対応は「その他」とします。

In [12]:
# 広域の地域マッピング（グループ分け用）
REGION_BROAD = {
    'United States of America': '北米', 'Canada': '北米', 'Mexico': '北米',
    'Belize': '中南米', 'Guatemala': '中南米', 'Honduras': '中南米', 'El Salvador': '中南米',
    'Nicaragua': '中南米', 'Costa Rica': '中南米', 'Panama': '中南米',
    'Cuba': '中南米', 'Jamaica': '中南米', 'Haiti': '中南米', 'Dominican Republic': '中南米',
    'Trinidad and Tobago': '中南米', 'Bahamas': '中南米', 'Barbados': '中南米',
    'Saint Lucia': '中南米', 'Grenada': '中南米', 'Antigua and Barbuda': '中南米',
    'Saint Vincent and the Grenadines': '中南米', 'Saint Kitts and Nevis': '中南米',
    'Dominica': '中南米', 'Suriname': '中南米', 'Guyana': '中南米',
    'Colombia': '中南米', 'Venezuela (Bolivarian Republic of)': '中南米',
    'Ecuador': '中南米', 'Peru': '中南米', 'Brazil': '中南米',
    'Bolivia (Plurinational State of)': '中南米', 'Chile': '中南米',
    'Argentina': '中南米', 'Uruguay': '中南米', 'Paraguay': '中南米',
    'Australia': 'オセアニア', 'New Zealand': 'オセアニア', 'Fiji': 'オセアニア',
    'Papua New Guinea': 'オセアニア', 'Solomon Islands': 'オセアニア', 'Vanuatu': 'オセアニア',
    'Samoa': 'オセアニア', 'Tonga': 'オセアニア', 'Kiribati': 'オセアニア',
    'Micronesia (Federated States of)': 'オセアニア', 'Marshall Islands': 'オセアニア', 'Palau': 'オセアニア',
    'Cook Islands': 'オセアニア', 'Nauru': 'オセアニア', 'Tuvalu': 'オセアニア',
    'Japan': 'アジア', 'China': 'アジア', 'India': 'アジア', 'Indonesia': 'アジア', 'Thailand': 'アジア',
    'Viet Nam': 'アジア', 'Philippines': 'アジア', 'Myanmar': 'アジア', 'Bangladesh': 'アジア',
    'Pakistan': 'アジア', 'Republic of Korea': 'アジア', 'Sri Lanka': 'アジア', 'Nepal': 'アジア',
    'Afghanistan': 'アジア', 'Cambodia': 'アジア', "Lao People's Democratic Republic": 'アジア',
    'Malaysia': 'アジア', 'Mongolia': 'アジア', 'Bhutan': 'アジア', 'Brunei Darussalam': 'アジア',
    'Timor-Leste': 'アジア', 'Democratic People\'s Republic of Korea': 'アジア',
    'France': 'ヨーロッパ', 'Germany': 'ヨーロッパ', 'United Kingdom of Great Britain and Northern Ireland': 'ヨーロッパ',
    'Italy': 'ヨーロッパ', 'Spain': 'ヨーロッパ', 'Poland': 'ヨーロッパ', 'Romania': 'ヨーロッパ',
    'Netherlands': 'ヨーロッパ', 'Belgium': 'ヨーロッパ', 'Greece': 'ヨーロッパ', 'Portugal': 'ヨーロッパ',
    'Czechia': 'ヨーロッパ', 'Hungary': 'ヨーロッパ', 'Austria': 'ヨーロッパ', 'Switzerland': 'ヨーロッパ',
    'Sweden': 'ヨーロッパ', 'Denmark': 'ヨーロッパ', 'Finland': 'ヨーロッパ', 'Norway': 'ヨーロッパ',
    'Ireland': 'ヨーロッパ', 'Russian Federation': 'ヨーロッパ', 'Ukraine': 'ヨーロッパ',
    'Albania': 'ヨーロッパ', 'Bosnia and Herzegovina': 'ヨーロッパ', 'Bulgaria': 'ヨーロッパ',
    'Croatia': 'ヨーロッパ', 'Serbia': 'ヨーロッパ', 'Slovakia': 'ヨーロッパ', 'Slovenia': 'ヨーロッパ',
    'North Macedonia': 'ヨーロッパ', 'Belarus': 'ヨーロッパ', 'Estonia': 'ヨーロッパ',
    'Latvia': 'ヨーロッパ', 'Lithuania': 'ヨーロッパ', 'Republic of Moldova': 'ヨーロッパ',
    'Montenegro': 'ヨーロッパ', 'Armenia': 'ヨーロッパ', 'Azerbaijan': 'ヨーロッパ',
    'Georgia': 'ヨーロッパ', 'Kazakhstan': 'ヨーロッパ', 'Kyrgyzstan': 'ヨーロッパ',
    'Tajikistan': 'ヨーロッパ', 'Turkmenistan': 'ヨーロッパ', 'Uzbekistan': 'ヨーロッパ',
    'Türkiye': 'ヨーロッパ', 'Cyprus': 'ヨーロッパ', 'Malta': 'ヨーロッパ',
    'Iceland': 'ヨーロッパ', 'Luxembourg': 'ヨーロッパ',
    'Nigeria': 'アフリカ', 'Ethiopia': 'アフリカ', 'Egypt': 'アフリカ', 'Democratic Republic of the Congo': 'アフリカ',
    'Tanzania': 'アフリカ', 'South Africa': 'アフリカ', 'Kenya': 'アフリカ', 'Uganda': 'アフリカ',
    'Algeria': 'アフリカ', 'Morocco': 'アフリカ', 'Sudan': 'アフリカ', 'Mozambique': 'アフリカ',
    'Ghana': 'アフリカ', 'Madagascar': 'アフリカ', 'Cameroon': 'アフリカ', "Côte d'Ivoire": 'アフリカ',
    'Niger': 'アフリカ', 'Burkina Faso': 'アフリカ', 'Mali': 'アフリカ', 'Malawi': 'アフリカ',
    'Zambia': 'アフリカ', 'Zimbabwe': 'アフリカ', 'Senegal': 'アフリカ', 'Chad': 'アフリカ',
    'Guinea': 'アフリカ', 'Rwanda': 'アフリカ', 'Benin': 'アフリカ', 'Tunisia': 'アフリカ',
    'Burundi': 'アフリカ', 'Somalia': 'アフリカ', 'Sierra Leone': 'アフリカ', 'Togo': 'アフリカ',
    'Libya': 'アフリカ', 'Liberia': 'アフリカ', 'Mauritania': 'アフリカ', 'Central African Republic': 'アフリカ',
    'Eritrea': 'アフリカ', 'Gambia': 'アフリカ', 'Botswana': 'アフリカ', 'Namibia': 'アフリカ',
    'Gabon': 'アフリカ', 'Lesotho': 'アフリカ', 'Guinea-Bissau': 'アフリカ', 'Mauritius': 'アフリカ',
    'Eswatini': 'アフリカ', 'Djibouti': 'アフリカ', 'Réunion': 'アフリカ', 'Comoros': 'アフリカ',
    'Cabo Verde': 'アフリカ', 'Equatorial Guinea': 'アフリカ', 'Congo': 'アフリカ',
    'Angola': 'アフリカ', 'United Republic of Tanzania': 'アフリカ',
    'Iran (Islamic Republic of)': '中東', 'Iraq': '中東', 'Saudi Arabia': '中東', 'Yemen': '中東',
    'Syrian Arab Republic': '中東', 'Jordan': '中東', 'Israel': '中東', 'Lebanon': '中東',
    'United Arab Emirates': '中東', 'Kuwait': '中東', 'Oman': '中東', 'Qatar': '中東', 'Bahrain': '中東',
    'Palestine': '中東',
}

# アラブ共和国エジプトなど表記ゆれ
REGION_BROAD['Arab Republic of Egypt'] = 'アフリカ'
regressionDF['RegionBroad'] = regressionDF['Area'].map(REGION_BROAD).fillna('その他')

## 高精度でなかった国ごとの指標

各国の観測数・収量の平均・標準偏差・気候変数（SPEI, GSL, Hurs, TXX）の平均を算出します。

In [ ]:
df_low = regressionDF[regressionDF['Area'].isin(areas_low)].copy()
df_low['RegionBroad'] = df_low['Area'].map(REGION_BROAD).fillna('その他')

agg_by_area = df_low.groupby('Area').agg(
    観測数=('Yield', 'count'),
    収量_平均=('Yield', 'mean'),
    収量_標準偏差=('Yield', 'std'),
    SPEI_平均=('SPEI', 'mean'),
    GSL_平均=('GSL', 'mean'),
    Hurs_平均=('Hurs', 'mean'),
    TXX_平均=('TXX', 'mean'),
    穀物数=('Item', 'nunique'),
).round(4)
agg_by_area = agg_by_area.reset_index()
agg_by_area['RegionBroad'] = agg_by_area['Area'].map(REGION_BROAD).fillna('その他')
agg_by_area

,Area,観測数,収量_平均,収量_標準偏差,SPEI_平均,GSL_平均,Hurs_平均,TXX_平均,穀物数,RegionBroad
0,Afghanistan,159,1718.6277,593.5283,-0.2938,268.1755,46.6519,35.2268,3,アジア
1,Albania,139,3011.0496,1158.7981,0.2760,296.0986,72.5225,30.9178,3,ヨーロッパ
2,Algeria,159,1706.3962,1158.4194,0.2062,359.3766,30.6709,43.5147,3,アフリカ
3,Angola,156,908.3365,356.7745,0.1200,360.0000,60.4044,34.1027,3,アフリカ
4,Antigua and Barbuda,47,1834.2170,282.1521,0.5238,360.0000,79.6481,28.2653,1,中南米
...,...,...,...,...,...,...,...,...,...,...
86,Ukraine,66,3629.8167,1085.2149,-0.0282,224.5082,74.6400,33.5545,3,ヨーロッパ
87,United Arab Emirates,56,10717.0607,10152.0033,-0.2813,360.0000,47.4905,44.5202,2,中東
88,Uzbekistan,66,3995.1500,1868.2706,-0.4027,256.9486,54.0900,40.6950,3,ヨーロッパ
89,Vanuatu,51,514.3039,29.9285,0.0971,360.0000,80.6224,28.3861,1,オセアニア


In [14]:
# 発展途上国＝国連・後発開発途上国（LDC）のみに限定 → グループ5が「ほぼ全員」にならないようにする
# データの Area 名に合わせた表記（Cabo Verde, Lao People's Democratic Republic 等）
# 参考: https://www.un.org/ohrlls/content/list-ldcs
DEVELOPING_COUNTRIES_LDC = {
    'Afghanistan', 'Angola', 'Benin', 'Burkina Faso', 'Burundi', 'Cambodia',
    'Central African Republic', 'Chad', 'Comoros', 'Democratic Republic of the Congo',
    'Djibouti', 'Eritrea', 'Ethiopia', 'Gambia', 'Guinea', 'Guinea-Bissau', 'Haiti',
    "Lao People's Democratic Republic", 'Lesotho', 'Liberia', 'Madagascar', 'Malawi',
    'Mali', 'Mauritania', 'Mozambique', 'Myanmar', 'Nepal', 'Niger', 'Rwanda', 'Senegal',
    'Sierra Leone', 'Solomon Islands', 'Somalia', 'South Sudan', 'Sudan', 'Timor-Leste',
    'Togo', 'Uganda', 'United Republic of Tanzania', 'Yemen', 'Zambia',
    'Cabo Verde', 'Kiribati', 'Tuvalu',
}
agg_by_area['発展途上国'] = agg_by_area['Area'].isin(DEVELOPING_COUNTRIES_LDC)
n_dev = agg_by_area['発展途上国'].sum()
print(f"発展途上国（LDCのみ）: {n_dev} ヶ国 → グループ5")
print(f"それ以外: {len(agg_by_area)-n_dev} ヶ国 → K-meansで4クラスタ（1〜4）")

発展途上国（LDCのみ）: 27 ヶ国 → グループ5
それ以外: 64 ヶ国 → K-meansで4クラスタ（1〜4）


---
## データに基づくグループ分類（クラスタリング）

- **発展途上国**（上で定義したリスト）は **グループ5に固定**。
- **それ以外**は観測数・収量のばらつき・気候変数（SPEI, GSL, Hurs, TXX 等）で **K-means を4クラスタ** に分類し、観測数少→多の順にグループ1〜4を付与。  
これにより旧「クラスター4（大国・地域差大）」のごちゃ混ぜを避け、発展途上国を明示した5グループになる。

In [ ]:
# 発展途上国はグループ5に固定。それ以外は K-means で4クラスタ（クラスター4の「ごみ」を解消）
import os
os.environ['OMP_NUM_THREADS'] = '1'  # Windows+MKL の KMeans メモリリーク警告を回避（sklearn 読み込み前に設定）
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

feat_cols = ['観測数', '収量_標準偏差', '収量_平均', 'SPEI_平均', 'GSL_平均', 'Hurs_平均', 'TXX_平均', '穀物数']
mask_dev = agg_by_area['発展途上国']

# 非発展途上国だけ 4 クラスタに分類
X = agg_by_area.loc[~mask_dev, feat_cols].fillna(agg_by_area[feat_cols].median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
km = KMeans(n_clusters=4, random_state=42, n_init=10)
agg_by_area.loc[~mask_dev, '_cluster'] = km.fit_predict(X_scaled)
agg_by_area.loc[mask_dev, '_cluster'] = -1  # 発展途上国は仮の-1

# クラスタ0..3の重心を観測数でソートし、1..4 を付与
centroids = agg_by_area[~mask_dev].groupby('_cluster')[feat_cols].mean()
rank = centroids['観測数'].rank().astype(int)
cluster_to_code = {c: int(rank.loc[c]) for c in centroids.index}
agg_by_area['GroupCode'] = 0
agg_by_area.loc[~mask_dev, 'GroupCode'] = agg_by_area.loc[~mask_dev, '_cluster'].map(cluster_to_code)
agg_by_area.loc[mask_dev, 'GroupCode'] = 5  # 発展途上国＝5

# 1〜4: データ駆動（観測数少→多）、5: 発展途上国
# ラベルはクラスタの実際の構成（国・代表値）に基づいて命名
GROUP_LABELS = {
    1: '1.観測数少・小規模',
    2: '2.収量変動大・不安定',
    3: '3.島嶼・沿岸・中規模（湿度高）',
    4: '4.急成長・新興国',
    5: '5.発展途上国（LDC）',
    0: 'その他',
}
agg_by_area['GroupLabel'] = agg_by_area['GroupCode'].map(GROUP_LABELS)
agg_by_area.drop(columns=['_cluster'], inplace=True)

print("=== 5グループ分類（非発展途上国はK-means 4クラスタ、発展途上国はグループ5固定） ===")
cnt = agg_by_area.groupby('GroupLabel')['Area'].count()
print(cnt.to_string())
print(f"\n合計: {cnt.sum()} ヶ国")

# グループ別の詳細な代表値（表）
detail = agg_by_area[agg_by_area['GroupCode'] > 0].groupby('GroupLabel').agg(
    国数=('Area', 'count'),
    観測数_平均=('観測数', 'mean'),
    観測数_最小=('観測数', 'min'),
    観測数_最大=('観測数', 'max'),
    収量SD_平均=('収量_標準偏差', 'mean'),
    収量SD_最小=('収量_標準偏差', 'min'),
    収量SD_最大=('収量_標準偏差', 'max'),
    収量_平均の平均=('収量_平均', 'mean'),
    SPEI_平均=('SPEI_平均', 'mean'),
    GSL_平均=('GSL_平均', 'mean'),
    Hurs_平均=('Hurs_平均', 'mean'),
    TXX_平均=('TXX_平均', 'mean'),
    穀物数_平均=('穀物数', 'mean'),
).round(2)
order = [GROUP_LABELS[i] for i in range(1, 6) if GROUP_LABELS[i] in detail.index]
detail = detail.reindex(order)
print("\n【グループ別の代表値（詳細）】")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
print(detail.to_string())

print("\n【グループ別の代表値（簡易・共通データの目安）】")
for g in range(1, 6):
    sub = agg_by_area[agg_by_area['GroupCode']==g][feat_cols]
    if len(sub) == 0:
        continue
    print(f"  {GROUP_LABELS[g]}: 観測数={sub['観測数'].mean():.0f}, 収量SD={sub['収量_標準偏差'].mean():.0f}, Hurs={sub['Hurs_平均'].mean():.1f}")

print("\n【グループ別・該当国一覧】")
for g in range(1, 6):
    sub = agg_by_area[agg_by_area['GroupCode']==g][['Area', '観測数', '収量_標準偏差']].sort_values('Area')
    if len(sub) == 0:
        continue
    print(f"\n  {GROUP_LABELS[g]} ({len(sub)} ヶ国)")
    for _, row in sub.iterrows():
        print(f"    - {row['Area']}: 観測数={row['観測数']}, 収量SD={row['収量_標準偏差']:.0f}")

=== 5グループ分類（非発展途上国はK-means 4クラスタ、発展途上国はグループ5固定） ===
GroupLabel
1.観測数少・小規模          18
2.収量変動大・不安定          2
3.島嶼・沿岸・中規模（湿度高）    23
4.急成長・新興国           21
5.発展途上国（LDC）        27

合計: 91 ヶ国

【グループ別の代表値（詳細）】
                  国数  観測数_平均  観測数_最小  観測数_最大  収量SD_平均  収量SD_最小   収量SD_最大  収量_平均の平均  SPEI_平均  GSL_平均  Hurs_平均  TXX_平均  穀物数_平均
GroupLabel                                                                                                                  
1.観測数少・小規模        18   49.56      16     159  1055.72   320.10   2840.07   3787.85    -0.27  223.56    76.41   28.37    1.78
2.収量変動大・不安定        2   55.00      54      56  8357.92  6563.84  10152.00   8828.94     0.16  360.00    64.34   36.35    2.00
3.島嶼・沿岸・中規模（湿度高）  23   79.83      47     106   764.54    29.93   1985.51   1933.31     0.10  358.94    77.25   31.44    1.57
4.急成長・新興国         21  135.95      66     159  1145.14   427.29   1977.57   2348.98    -0.09  328.10    57.19   37.82    2.90
5.発展途上国（LDC）      27  101.37       2     159

In [16]:
# 各グループの「共通するデータ」＝グループ内で共通する数値特性（言い訳の根拠）
if 'GroupCode' not in agg_by_area.columns:
    raise RuntimeError('GroupCode がありません。直上の「データに基づく5グループ分類（K-means）」のセルを先に実行してください。')
group_common = agg_by_area[agg_by_area['GroupCode']>0].groupby('GroupLabel').agg(
    国数=('Area', 'count'),
    観測数_平均=('観測数', 'mean'),
    観測数_最小=('観測数', 'min'),
    観測数_最大=('観測数', 'max'),
    収量標準偏差_平均=('収量_標準偏差', 'mean'),
    収量_平均の平均=('収量_平均', 'mean'),
    SPEI_平均=('SPEI_平均', 'mean'),
    GSL_平均=('GSL_平均', 'mean'),
    Hurs_平均=('Hurs_平均', 'mean'),
    TXX_平均=('TXX_平均', 'mean'),
    穀物数_平均=('穀物数', 'mean'),
).round(2)
order = [GROUP_LABELS[i] for i in range(1,6) if GROUP_LABELS[i] in group_common.index]
group_common = group_common.reindex(order)
print("=== 各グループに共通するデータ特性（学会で「こういう国ばかり」と示す用） ===")
group_common

=== 各グループに共通するデータ特性（学会で「こういう国ばかり」と示す用） ===


,国数,観測数_平均,観測数_最小,観測数_最大,収量標準偏差_平均,収量_平均の平均,SPEI_平均,GSL_平均,Hurs_平均,TXX_平均,穀物数_平均
GroupLabel,,,,,,,,,,,
1.観測数少・小規模,18,49.56,16,159,1055.72,3787.85,-0.27,223.56,76.41,28.37,1.78
2.収量変動大・不安定,2,55.00,54,56,8357.92,8828.94,0.16,360.00,64.34,36.35,2.00
3.島嶼・沿岸・中規模（湿度高）,23,79.83,47,106,764.54,1933.31,0.10,358.94,77.25,31.44,1.57
4.急成長・新興国,21,135.95,66,159,1145.14,2348.98,-0.09,328.10,57.19,37.82,2.90
5.発展途上国（LDC）,27,101.37,2,159,555.86,1462.31,-0.13,352.54,61.26,35.79,2.22


In [17]:
# スライド用：分類・該当国例・原因の推察（＋共通データの要約）
# データから得られたグループ特徴に基づく解釈
if 'GroupCode' not in agg_by_area.columns:
    raise RuntimeError('GroupCode がありません。直上の「データに基づく5グループ分類（K-means）」のセルを先に実行してください。')
REASON = {
    1: '観測数が少なく統計的に不安定（小規模経済・欧州・旧ソ連等）',
    2: '収量の年次変動が極端に大きくトレンド推定が困難',
    3: '島嶼・沿岸国が多く湿度高、観測数・収量変動は中程度で気候-収量の関係が多様',
    4: '急成長・新興国が多く、経済・農業構造の変化が大きくトレンド推定が難しい',
    5: '後発開発途上国（LDC）・データ整備や気候以外の要因がモデル適合を妨げやすい',
}
slide_rows = []
for gcode in [1,2,3,4,5]:
    if gcode not in agg_by_area['GroupCode'].values:
        continue
    label = GROUP_LABELS[gcode]
    countries = agg_by_area[agg_by_area['GroupCode']==gcode]['Area'].tolist()
    examples = ', '.join(sorted(countries)[:5]) + ('...' if len(countries)>5 else '')
    r = group_common.loc[label]
    data_note = f"観測数平均{r['観測数_平均']:.0f}, 収量SD平均{r['収量標準偏差_平均']:.0f}"
    slide_rows.append({
        '分類': label.replace(str(gcode)+'.',''),
        '該当国数': len(countries),
        '該当国の例': examples,
        '原因の推察': REASON[gcode],
        '共通データの特徴': data_note,
    })
slide_df = pd.DataFrame(slide_rows)
print("=== スライド用：本モデルが適合しなかった国々の特徴分析 ===\n")
print(slide_df.to_string(index=False))

=== スライド用：本モデルが適合しなかった国々の特徴分析 ===

            分類  該当国数                                                              該当国の例                                  原因の推察             共通データの特徴
      観測数少・小規模    18       Bhutan, Bosnia and Herzegovina, Croatia, Denmark, Estonia...          観測数が少なく統計的に不安定（小規模経済・欧州・旧ソ連等）  観測数平均50, 収量SD平均1056
     収量変動大・不安定     2             Saint Vincent and the Grenadines, United Arab Emirates                収量の年次変動が極端に大きくトレンド推定が困難  観測数平均55, 収量SD平均8358
島嶼・沿岸・中規模（湿度高）    23 Antigua and Barbuda, Barbados, Belize, Brunei Darussalam, Congo...  島嶼・沿岸国が多く湿度高、観測数・収量変動は中程度で気候-収量の関係が多様   観測数平均80, 収量SD平均765
       急成長・新興国    21                  Albania, Algeria, Azerbaijan, Botswana, Brazil...    急成長・新興国が多く、経済・農業構造の変化が大きくトレンド推定が難しい 観測数平均136, 収量SD平均1145
    発展途上国（LDC）    27            Afghanistan, Angola, Benin, Burkina Faso, Cabo Verde... 後発開発途上国（LDC）・データ整備や気候以外の要因がモデル適合を妨げやすい  観測数平均101, 収量SD平均556


### 各グループの解釈（Q&A用）

上記の**「共通するデータ」**の表で、各グループの数値的な特徴（観測数・収量変動・気候変数）を確認できる。スライド用サマリーの**「原因の推察」**は、そのデータに基づく解釈である。発表時は「このグループは観測数が少なく〜」「このグループは収量のばらつきが大きく〜」のように、**共通するデータ**を根拠に説明する。

### 結論（スライドの締め）

**「精度が低かった91カ国」** ではなく、次のように言い換える：

> **「本モデルが適合しなかった国々の特徴分析」** の結果、クラスタリングにより**データの類似性に基づく5つのグループ**に分かれた。各グループは観測数・収量変動・気候特性などで共通するデータを持ち、特異点（Outliers）として「一般的な回帰モデルには馴染まない構造」があることが示された。  
> **これらを除いた65カ国においては本モデルは高い精度を示しており、『モデルが有効な国』と『適用限界のある国』の棲み分けが明確になった**と言える。

In [18]:
# グループごとの国一覧（発表用）
if 'GroupCode' not in agg_by_area.columns:
    raise RuntimeError('GroupCode がありません。「データに基づく5グループ分類（K-means）」のセルを先に実行してください。')
for gcode in [1,2,3,4,5,0]:
    label = GROUP_LABELS[gcode]
    sub = agg_by_area[agg_by_area['GroupCode']==gcode]
    if len(sub)==0:
        continue
    countries = sub['Area'].tolist()
    print(f"\n【{label}】 {len(countries)} ヶ国")
    for c in sorted(countries):
        obs = sub[sub['Area']==c]['観測数'].values[0]
        sd = sub[sub['Area']==c]['収量_標準偏差'].values[0]
        print(f"  - {c}  (観測数={obs}, 収量SD={sd:.0f})")


【1.観測数少・小規模】 18 ヶ国
  - Bhutan  (観測数=159, 収量SD=666)
  - Bosnia and Herzegovina  (観測数=44, 収量SD=766)
  - Croatia  (観測数=44, 収量SD=1105)
  - Denmark  (観測数=57, 収量SD=1216)
  - Estonia  (観測数=22, 収量SD=635)
  - Finland  (観測数=53, 収量SD=745)
  - Georgia  (観測数=44, 収量SD=554)
  - Ireland  (観測数=53, 収量SD=2141)
  - Latvia  (観測数=22, 収量SD=642)
  - Lithuania  (観測数=44, 収量SD=1606)
  - Luxembourg  (観測数=28, 収量SD=1367)
  - Mongolia  (観測数=53, 収量SD=320)
  - Montenegro  (観測数=16, 収量SD=661)
  - Norway  (観測数=53, 収量SD=750)
  - Serbia  (観測数=16, 収量SD=917)
  - Sweden  (観測数=52, 収量SD=987)
  - Tajikistan  (観測数=66, 収量SD=2840)
  - Ukraine  (観測数=66, 収量SD=1085)

【2.収量変動大・不安定】 2 ヶ国
  - Saint Vincent and the Grenadines  (観測数=54, 収量SD=6564)
  - United Arab Emirates  (観測数=56, 収量SD=10152)

【3.島嶼・沿岸・中規模（湿度高）】 23 ヶ国
  - Antigua and Barbuda  (観測数=47, 収量SD=282)
  - Barbados  (観測数=52, 収量SD=264)
  - Belize  (観測数=106, 収量SD=1016)
  - Brunei Darussalam  (観測数=53, 収量SD=550)
  - Congo  (観測数=106, 収量SD=334)
  - Cuba  (観測数=104, 収量SD=884)
  - Cyprus

In [19]:
# 学会発表用の分類結果をCSVで保存
if 'GroupCode' not in agg_by_area.columns:
    raise RuntimeError('GroupCode がありません。「データに基づく5グループ分類（K-means）」のセルを先に実行してください。')
out = agg_by_area[['Area', 'GroupLabel', 'GroupCode', '発展途上国', '観測数', '収量_平均', '収量_標準偏差',
                   'SPEI_平均', 'GSL_平均', 'Hurs_平均', 'TXX_平均', '穀物数']].copy()
out.columns = ['国', 'グループ', 'GroupCode', '発展途上国', '観測数', '収量_平均', '収量_標準偏差',
               'SPEI_平均', 'GSL_平均', 'Hurs_平均', 'TXX_平均', '穀物数']
out.sort_values(['GroupCode', '国']).to_csv(
    os.path.join(base_dir, 'low_accuracy_countries_5groups.csv'), index=False, encoding='utf-8-sig')
group_common.to_csv(os.path.join(base_dir, 'low_accuracy_5groups_common_data.csv'), encoding='utf-8-sig')
slide_df.to_csv(os.path.join(base_dir, 'low_accuracy_slide_summary.csv'), index=False, encoding='utf-8-sig')
print("✓ low_accuracy_countries_5groups.csv  （国×グループ×指標）")
print("✓ low_accuracy_5groups_common_data.csv （グループ別の共通データ特性）")
print("✓ low_accuracy_slide_summary.csv       （スライド用 分類・該当国例・原因）")

✓ low_accuracy_countries_5groups.csv  （国×グループ×指標）
✓ low_accuracy_5groups_common_data.csv （グループ別の共通データ特性）
✓ low_accuracy_slide_summary.csv       （スライド用 分類・該当国例・原因）


## 5グループの世界地図

高精度でなかった国を5グループに色分けした世界地図（65ヶ国は対象外のためグレー表示）。

In [ ]:
# 高精度でなかった91ヶ国の国名 → ISO3（Plotly Choropleth用）
ISO3_LOW = {
    'Afghanistan': 'AFG', 'Albania': 'ALB', 'Algeria': 'DZA', 'Angola': 'AGO',
    'Antigua and Barbuda': 'ATG', 'Azerbaijan': 'AZE', 'Barbados': 'BRB', 'Belize': 'BLZ',
    'Benin': 'BEN', 'Bhutan': 'BTN', 'Bosnia and Herzegovina': 'BIH', 'Botswana': 'BWA',
    'Brazil': 'BRA', 'Brunei Darussalam': 'BRN', 'Bulgaria': 'BGR', 'Burkina Faso': 'BFA',
    'Cabo Verde': 'CPV', 'Cambodia': 'KHM', 'Cameroon': 'CMR', 'Central African Republic': 'CAF',
    'Chad': 'TCD', 'Congo': 'COG', 'Croatia': 'HRV', 'Cuba': 'CUB', 'Cyprus': 'CYP',
    "Côte d'Ivoire": 'CIV', 'Denmark': 'DNK', 'Djibouti': 'DJI', 'Dominica': 'DMA',
    'Eritrea': 'ERI', 'Estonia': 'EST', 'Ethiopia': 'ETH', 'Finland': 'FIN', 'Gabon': 'GAB',
    'Georgia': 'GEO', 'Grenada': 'GRD', 'Guinea': 'GIN', 'Hungary': 'HUN', 'India': 'IND',
    'Indonesia': 'IDN', 'Iraq': 'IRQ', 'Ireland': 'IRL', 'Jamaica': 'JAM', 'Jordan': 'JOR',
    'Kenya': 'KEN', "Lao People's Democratic Republic": 'LAO', 'Latvia': 'LVA', 'Lebanon': 'LBN',
    'Lesotho': 'LSO', 'Liberia': 'LBR', 'Lithuania': 'LTU', 'Luxembourg': 'LUX',
    'Madagascar': 'MDG', 'Malawi': 'MWI', 'Malaysia': 'MYS', 'Maldives': 'MDV', 'Mali': 'MLI',
    'Malta': 'MLT', 'Mauritius': 'MUS', 'Mongolia': 'MNG', 'Montenegro': 'MNE', 'Mozambique': 'MOZ',
    'Nepal': 'NPL', 'Nigeria': 'NGA', 'Norway': 'NOR', 'Pakistan': 'PAK', 'Papua New Guinea': 'PNG',
    'Paraguay': 'PRY', 'Philippines': 'PHL', 'Romania': 'ROU', 'Saint Vincent and the Grenadines': 'VCT',
    'Saudi Arabia': 'SAU', 'Senegal': 'SEN', 'Serbia': 'SRB', 'Sierra Leone': 'SLE',
    'Solomon Islands': 'SLB', 'South Africa': 'ZAF', 'South Sudan': 'SSD', 'Sri Lanka': 'LKA',
    'Sudan': 'SDN', 'Sweden': 'SWE', 'Tajikistan': 'TJK', 'Timor-Leste': 'TLS', 'Tunisia': 'TUN',
    'Turkmenistan': 'TKM', 'Uganda': 'UGA', 'Ukraine': 'UKR', 'United Arab Emirates': 'ARE',
    'Uzbekistan': 'UZB', 'Vanuatu': 'VUT', 'Zimbabwe': 'ZWE',
}
import plotly.graph_objects as go
if 'GroupCode' not in agg_by_area.columns:
    raise RuntimeError('GroupCode がありません。「データに基づく5グループ分類（K-means）」のセルを先に実行してください。')
map_df = agg_by_area[agg_by_area['GroupCode'] > 0][['Area', 'GroupCode', 'GroupLabel']].copy()
map_df['ISO3'] = map_df['Area'].map(ISO3_LOW)
map_df = map_df.dropna(subset=['ISO3'])
# 離散色: グループ1=青, 2=橙, 3=緑, 4=紫, 5=茶。凡例は上から1→5になるよう反転
# zを 6-GroupCode にすると地図の色はそのままに、色バーが上=1・下=5になる
order_codes = list(range(1, 6))
colorscale_reversed = [[0, '#795548'], [0.2, '#795548'], [0.2, '#9b59b6'], [0.4, '#9b59b6'],
                       [0.4, '#27ae60'], [0.6, '#27ae60'], [0.6, '#e67e22'], [0.8, '#e67e22'],
                       [0.8, '#3498db'], [1, '#3498db']]
z_display = 6 - map_df['GroupCode']
fig = go.Figure(go.Choropleth(
    locations=map_df['ISO3'],
    z=z_display,
    locationmode='ISO-3',
    colorscale=colorscale_reversed,
    zmin=1, zmax=5,
    colorbar=dict(
        title='グループ',
        len=0.5,
        tickvals=[1.4, 2.2, 3.0, 3.8, 4.6],
        ticktext=[GROUP_LABELS[i] for i in reversed(order_codes)],
        tickmode='array',
    ),
    hovertext=[f"{a}<br>{GROUP_LABELS[g]}" for a, g in zip(map_df['Area'], map_df['GroupCode'])],
    hoverinfo='text',
))
fig.update_geos(showcoastlines=True, landcolor='#bdc3c7', showcountries=True, countrycolor='#7f8c8d')
fig.update_layout(
    title='高精度でなかった国の5グループ分類（世界地図）',
    geo=dict(showframe=False),
    height=500, margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

: 

## グループ分け1：地域（RegionBroad）

高精度でなかった国を地域ごとに集計します。

In [21]:
group_region = agg_by_area.groupby('RegionBroad').agg(
    国数=('Area', 'count'),
    観測数_合計=('観測数', 'sum'),
    観測数_平均=('観測数', 'mean'),
    収量平均_地域平均=('収量_平均', 'mean'),
).round(2)
group_region = group_region.sort_values('国数', ascending=False)
print("=== グループ分け1：地域別 ===")
print(group_region)
print("\n地域ごとの国一覧:")
for r in group_region.index:
    countries = agg_by_area[agg_by_area['RegionBroad'] == r]['Area'].tolist()
    print(f"\n  【{r}】 {len(countries)} ヶ国")
    for c in sorted(countries):
        print(f"    - {c}")

=== グループ分け1：地域別 ===
             国数  観測数_合計  観測数_平均  収量平均_地域平均
RegionBroad                               
アフリカ         32    3546  110.81    1526.72
ヨーロッパ        25    1599   63.96    3703.45
アジア          14    1606  114.71    1884.97
中南米          10     891   89.10    2376.35
中東            5     530  106.00    3803.78
オセアニア         3     205   68.33    2073.77
その他           2      53   26.50    1140.71

地域ごとの国一覧:

  【アフリカ】 32 ヶ国
    - Algeria
    - Angola
    - Benin
    - Botswana
    - Burkina Faso
    - Cabo Verde
    - Cameroon
    - Central African Republic
    - Chad
    - Congo
    - Côte d'Ivoire
    - Djibouti
    - Eritrea
    - Ethiopia
    - Gabon
    - Guinea
    - Kenya
    - Lesotho
    - Liberia
    - Madagascar
    - Malawi
    - Mali
    - Mauritius
    - Mozambique
    - Nigeria
    - Senegal
    - Sierra Leone
    - South Africa
    - Sudan
    - Tunisia
    - Uganda
    - Zimbabwe

  【ヨーロッパ】 25 ヶ国
    - Albania
    - Azerbaijan
    - Bosnia and Herzegovina
    - B

## グループ分け2：データ量（観測数）

観測数が少ない国は予測が不安定になりやすいと仮定し、少・中・多でグループ分けします。

In [22]:
q1 = agg_by_area['観測数'].quantile(0.33)
q2 = agg_by_area['観測数'].quantile(0.67)
def obs_group(n):
    if n <= q1: return '観測数_少'
    if n <= q2: return '観測数_中'
    return '観測数_多'
agg_by_area['グループ_データ量'] = agg_by_area['観測数'].apply(obs_group)

group_data = agg_by_area.groupby('グループ_データ量').agg(
    国数=('Area', 'count'),
    観測数_最小=('観測数', 'min'),
    観測数_最大=('観測数', 'max'),
    収量_平均=('収量_平均', 'mean'),
).round(2)
print("=== グループ分け2：データ量（観測数） ===")
print(f"境界: 少 ≤{q1:.0f}, 中 ≤{q2:.0f}, 多 >{q2:.0f}")
print(group_data)
print("\nデータ量グループごとの国一覧:")
for g in ['観測数_少', '観測数_中', '観測数_多']:
    countries = agg_by_area[agg_by_area['グループ_データ量'] == g]['Area'].tolist()
    print(f"\n  【{g}】 {len(countries)} ヶ国")
    for c in sorted(countries):
        print(f"    - {c}")

=== グループ分け2：データ量（観測数） ===
境界: 少 ≤53, 中 ≤106, 多 >106
           国数  観測数_最小  観測数_最大    収量_平均
グループ_データ量                             
観測数_中      34      54     106  2547.64
観測数_多      26     126     159  2013.95
観測数_少      31       2      53  2584.95

データ量グループごとの国一覧:

  【観測数_少】 31 ヶ国
    - Antigua and Barbuda
    - Barbados
    - Bosnia and Herzegovina
    - Brunei Darussalam
    - Cabo Verde
    - Croatia
    - Cyprus
    - Djibouti
    - Dominica
    - Eritrea
    - Estonia
    - Finland
    - Georgia
    - Grenada
    - Ireland
    - Latvia
    - Liberia
    - Lithuania
    - Luxembourg
    - Maldives
    - Malta
    - Mongolia
    - Montenegro
    - Norway
    - Serbia
    - Solomon Islands
    - South Sudan
    - Sudan
    - Sweden
    - Tunisia
    - Vanuatu

  【観測数_中】 34 ヶ国
    - Azerbaijan
    - Belize
    - Benin
    - Botswana
    - Burkina Faso
    - Cambodia
    - Central African Republic
    - Congo
    - Cuba
    - Côte d'Ivoire
    - Denmark
    - Ethiopia
    - Gabon
    - 

## グループ分け3：収量水準

収量の平均が低い国・中程度・高い国に分けます。

In [23]:
y1 = agg_by_area['収量_平均'].quantile(0.33)
y2 = agg_by_area['収量_平均'].quantile(0.67)
def yield_group(y):
    if y <= y1: return '収量_低'
    if y <= y2: return '収量_中'
    return '収量_高'
agg_by_area['グループ_収量'] = agg_by_area['収量_平均'].apply(yield_group)

group_yield = agg_by_area.groupby('グループ_収量').agg(
    国数=('Area', 'count'),
    収量_最小=('収量_平均', 'min'),
    収量_最大=('収量_平均', 'max'),
    観測数_平均=('観測数', 'mean'),
).round(2)
print("=== グループ分け3：収量水準 ===")
print(f"境界: 低 ≤{y1:.0f}, 中 ≤{y2:.0f}, 高 >{y2:.0f}")
print(group_yield)

=== グループ分け3：収量水準 ===
境界: 低 ≤1676, 中 ≤2502, 高 >2502
         国数    収量_最小     収量_最大  観測数_平均
グループ_収量                               
収量_中     31  1687.64   2479.05  112.42
収量_低     30   389.49   1650.10   96.60
収量_高     30  2556.98  10717.06   68.23


## クロス集計：地域 × データ量

地域とデータ量の両方で見たグループ分け表です。

In [ ]:
cross = pd.crosstab(agg_by_area['RegionBroad'], agg_by_area['グループ_データ量'], margins=True)
print("=== 地域 × データ量（国数） ===")
print(cross)

summary = agg_by_area[['Area', 'RegionBroad', 'グループ_データ量', 'グループ_収量', '観測数', '収量_平均']].copy()
summary.columns = ['国', '地域', 'グループ_データ量', 'グループ_収量', '観測数', '収量_平均']
summary = summary.sort_values(['地域', '観測数'])
print("\n=== 高精度でなかった国 一覧（地域・グループ付き） ===")
summary

=== 地域 × データ量（国数） ===
グループ_データ量    観測数_中  観測数_多  観測数_少  All
RegionBroad                          
その他              0      0      2    2
アジア              7      5      2   14
アフリカ            13     13      6   32
オセアニア            1      0      2    3
ヨーロッパ            6      4     15   25
中南米              4      2      4   10
中東               3      2      0    5
All             34     26     31   91

=== 高精度でなかった国 一覧（地域・グループ付き） ===


,国,地域,グループ_データ量,グループ_収量,観測数,収量_平均
77,South Sudan,その他,観測数_少,収量_低,2,496.3000
55,Maldives,その他,観測数_少,収量_中,51,1785.1176
13,Brunei Darussalam,アジア,観測数_少,収量_低,53,1376.3868
59,Mongolia,アジア,観測数_少,収量_低,53,923.5925
54,Malaysia,アジア,観測数_中,収量_中,83,2392.3337
...,...,...,...,...,...,...
87,United Arab Emirates,中東,観測数_中,収量_高,56,10717.0607
43,Jordan,中東,観測数_中,収量_低,83,1633.5193
47,Lebanon,中東,観測数_中,収量_中,106,2010.2679
71,Saudi Arabia,中東,観測数_多,収量_高,126,2767.1286
